# Week 8 · Residual Learning & ResNet

skip connection의 shape와 gradient 경로를 확인하고 작은 residual MLP를 비교합니다.

[강의로 돌아가기](https://ml-fundamentals-karpnet.vercel.app/week/8)

**사용법**: 파일 → Drive에 사본 저장 → 위에서 아래로 실행하세요. 런타임을 다시 시작했다면 설정 셀부터 다시 실행합니다. CPU 기준의 짧은 실습이며 데이터 다운로드가 없습니다. 실행 결과는 사이트에 자동 저장되지 않습니다.


## 1. 실행 준비

Colab 기본 Python 런타임의 NumPy·Matplotlib·PyTorch를 사용합니다. import 오류가 나면 새 기본 런타임으로 연결하세요.


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__)
import torch
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(1)
device = torch.device('cpu')  # 기본 실습은 GPU가 필요 없습니다.
print('PyTorch:', torch.__version__, '| device:', device)


## 2. 실행 전 예상

F(x)=0일 때 x+F(x)의 출력은? plain block의 출력은? 더 깊은 모델이 항상 더 좋은 결과를 낼까요?

아래 칸에 예상과 이유를 먼저 적으세요.


**내 예상:**

(여기에 작성)


## 3. 실행하고 관찰하기

아래 parameter를 확인하고 두 코드 셀을 순서대로 실행하세요.


In [ ]:
depth = 6  # Apply: 2, 12 비교
steps = 100
learning_rate = 0.01


In [ ]:
x=torch.tensor([[1.,2.,3.]],requires_grad=True)
residual=x+0*x
residual.sum().backward()
print('F=0 output:',residual.detach(),'| gradient to x:',x.grad)
assert torch.allclose(x.grad,torch.ones_like(x))

class Block(nn.Module):
    def __init__(self,width,residual):
        super().__init__(); self.linear=nn.Linear(width,width); self.residual=residual
    def forward(self,x):
        f=torch.relu(self.linear(x))
        return x+f if self.residual else f

def make_model(residual):
    return nn.Sequential(nn.Linear(1,8),*[Block(8,residual) for _ in range(depth)],nn.Linear(8,1))

train_x=torch.linspace(-2,2,128).reshape(-1,1)
train_y=torch.sin(2*train_x)
results={}
for label,use_skip in [('plain',False),('residual',True)]:
    torch.manual_seed(42)  # 같은 형태의 parameter를 같은 초기값으로 비교
    model=make_model(use_skip)
    optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)
    history=[]
    for step in range(steps):
        optimizer.zero_grad(); loss=((model(train_x)-train_y)**2).mean()
        loss.backward(); optimizer.step(); history.append(loss.item())
    results[label]=history
    plt.plot(history,label=label)
plt.xlabel('step');plt.ylabel('training MSE');plt.legend();plt.show()
print({name:round(values[-1],4) for name,values in results.items()})


## 4. Apply · 한 가지씩 바꾸기

depth만 바꿔 학습 곡선을 비교하세요. 어떤 설정에서는 residual이 더 나쁠 수도 있습니다. 이 실험은 CNN ResNet 전체가 아닌 skip connection 원리 실험입니다.

parameter를 바꿀 때는 parameter 셀과 아래 실행 셀을 모두 다시 실행하세요.


| 바꾼 값 | 실행 전 예상 | 실제 결과 | 설명 |
|---|---|---|---|
| 기본값 | | | |
| 변경 1 | | | |
| 변경 2 | | | |


## 5. Explore · 선택 확장

CNN block의 F에 Conv2d 두 개를 사용해 보세요. 채널 수나 해상도가 바뀌면 x+F(x)를 위해 어떤 projection이 필요할까요? BatchNorm, 초기화, 학습률까지 통제하지 않고 성능 차이를 일반화할 수 있을까요?


In [ ]:
# 선택 확장 코드를 여기에 작성하세요.


## 6. 내 말로 설명하기

- 예상과 결과가 달랐던 점은?
- 이번 실습을 한 문장으로 설명하면?
- 아직 설명하기 어려운 부분은?

답을 자신의 Drive 사본에 남긴 뒤 [사이트로 돌아가 학습 기록을 체크](https://ml-fundamentals-karpnet.vercel.app/week/8)하세요. 노트북 실행만으로 주차 잠금이나 완료 기록이 자동 변경되지는 않습니다.
